# 05 - Voting and Stacking
Two ways to combine **diverse** base models: Random Forest (bagging), XGBoost and CatBoost (two different boosting libraries). Same split and log1p target as before; scores are on the original case scale.

In [1]:
# AI-GENERATED (Claude via Antigravity)
import sys, warnings, numpy as np, pandas as pd
sys.path.append("../src")
from split import *
from evaluate import regression_scores, fit_and_score
from stacking import make_base_models, TimeSeriesStacker
warnings.filterwarnings("ignore")
df = load_features("../data/processed/features.csv")
train, test = time_split(df)
features = feature_columns(df)
X_train, X_test = train[features], test[features]
y_train_log, y_train_raw, y_test_raw = train[TARGET_LOG], train[TARGET_RAW], test[TARGET_RAW]
cv = make_cv(train)                      # TimeSeriesSplit(5) with a one-week gap, as in step 03

## 1. Voting: just average
`VotingRegressor` fits each base model on the full training set and **averages** their predictions (here on the log scale, then `expm1`). It has no weights to learn, so it can't overfit its own combining step.

In [2]:
# AI-GENERATED (Claude via Antigravity)
from sklearn.ensemble import VotingRegressor
voting = VotingRegressor(list(make_base_models(features).items()))
rows = [fit_and_score("Voting (RF + XGB + CatBoost)", "combined", voting, X_train, y_train_log, X_test, y_train_raw, y_test_raw)]
print(rows[-1])

{'model': 'Voting (RF + XGB + CatBoost)', 'family': 'combined', 'train_RMSE': 16.87992686259341, 'test_RMSE': 26.797118941299036, 'test_MAE': 12.231341964691987, 'test_R2': 0.8174106761683635, 'train_seconds': 43.48491949995514}


## 2. Stacking: why scikit-learn's `StackingRegressor` does not work here
**Idea of stacking:** a small *meta-model* learns how much to trust each base model. To teach it fairly, it must see base-model predictions on rows those models **did not train on** (out-of-fold predictions); otherwise an overfit model (like a deep Random Forest) looks perfect and gets too much weight.

**The limitation:** `StackingRegressor` makes those predictions with `cross_val_predict`, which requires the folds to be a *partition*: every row predicted exactly once. `TimeSeriesSplit` does not do that. The earliest block is only ever used for training and is never in a validation part, so those rows would get no prediction. Random K-fold would satisfy the rule, but it shuffles time, which we ruled out. The cell below checks this.

In [3]:
# AI-GENERATED (Claude via Antigravity)
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.tree import DecisionTreeRegressor
Z = make_preprocessor(features).fit_transform(X_train)                     # numbers only, for a quick check
tiny = StackingRegressor([("tree", DecisionTreeRegressor(max_depth=3, random_state=42))], final_estimator=RidgeCV(), cv=cv)
try:
    tiny.fit(Z, y_train_log); print("It worked (unexpected)")
except Exception as e:
    print(type(e).__name__, "->", str(e)[:300])

ValueError -> cross_val_predict only works for partitions


**Manual stacking (`src/stacking.py`, class `TimeSeriesStacker`)**
1. For each `TimeSeriesSplit` fold: fit every base model on the fold's **past** part and predict its **future** part.
2. Collect those out-of-fold predictions. The earliest block has none, so it is left out of the meta-model's training.
3. Fit a `RidgeCV` meta-model on the out-of-fold predictions (its regularisation is also chosen with time-ordered folds).
4. Refit every base model on the **full** training set. At prediction time the meta-model combines these final models.

Cost: the base models are trained 5 (folds) + 1 (final) times each, so stacking is the slowest model here.

In [4]:
# AI-GENERATED (Claude via Antigravity)
stack = TimeSeriesStacker(make_base_models(features), cv)
rows.append(fit_and_score("Stacking (RF + XGB + CatBoost, RidgeCV)", "combined", stack, X_train, y_train_log, X_test, y_train_raw, y_test_raw))
print(rows[-1])
print("Meta-model trained on", stack.n_meta_rows_, "out-of-fold rows of", len(train), "(earliest block excluded)")

{'model': 'Stacking (RF + XGB + CatBoost, RidgeCV)', 'family': 'combined', 'train_RMSE': 14.795398653082104, 'test_RMSE': 27.282303861219336, 'test_MAE': 12.509349964053976, 'test_R2': 0.8107389453720948, 'train_seconds': 191.42043299996294}
Meta-model trained on 16165 out-of-fold rows of 19400 (earliest block excluded)


## What weights did the meta-model give each base model?
These are Ridge coefficients on the log scale. They are not forced to sum to 1 or to be positive; the intercept corrects any overall bias.

In [5]:
# AI-GENERATED (Claude via Antigravity)
w = pd.Series(stack.meta_.coef_, index=list(stack.base_models), name="weight")
display(w.round(3).to_frame().assign(share=lambda t: (t.weight / t.weight.sum()).round(2)))
print("intercept: %.3f | sum of weights: %.3f | chosen ridge alpha: %.3g" % (stack.meta_.intercept_, w.sum(), stack.meta_.alpha_))

,weight,share
Random Forest,0.646,0.64
XGBoost,-0.020,-0.02
CatBoost,0.389,0.38


intercept: 0.027 | sum of weights: 1.015 | chosen ridge alpha: 0.001


## Results table (default models from step 04 + the two combined models)

In [6]:
# AI-GENERATED (Claude via Antigravity)
old = pd.read_csv("../reports/results_default.csv")
old = old[old["family"] != "combined"].drop(columns="test/train RMSE", errors="ignore")      # re-running this notebook never duplicates rows
res = pd.concat([old, pd.DataFrame(rows)]).sort_values("test_RMSE").reset_index(drop=True)
res["test/train RMSE"] = res["test_RMSE"] / res["train_RMSE"]
res.to_csv("../reports/results_default.csv", index=False)
display(res.round(3))

,model,family,train_RMSE,test_RMSE,test_MAE,test_R2,train_seconds,test/train RMSE
0,Gradient Boosting,boosting,27.213,26.054,12.035,0.827,25.182,0.957
1,Voting (RF + XGB + CatBoost),combined,16.880,26.797,12.231,0.817,43.485,1.588
2,"Stacking (RF + XGB + CatBoost, RidgeCV)",combined,14.795,27.282,12.509,0.811,191.420,1.844
3,CatBoost,boosting,21.955,27.613,12.645,0.806,13.074,1.258
4,Baseline: persistence (this week),baseline,28.157,27.677,12.802,0.805,0.000,0.983
5,Random Forest,bagging,11.533,27.821,12.484,0.803,27.480,2.412
6,Bagging (decision trees),bagging,11.746,27.878,12.525,0.802,25.251,2.373
7,XGBoost,boosting,19.025,28.338,12.825,0.796,2.127,1.490
8,AdaBoost,boosting,61.763,41.286,19.563,0.567,8.827,0.668
9,Baseline: district training average,baseline,76.893,48.961,21.906,0.390,0.000,0.637
